# Robot 2 · un arbre de décision apprend à combattre

Le robot 1, vous avez écrit ses règles à l'intuition. Ici, les règles viennent des données : un **arbre de décision** regarde ce qu'ont fait les combattants qui ont **gagné**, situation par situation, et en tire des règles `if` / `else`.

Les données :
- `decisions.csv` : une ligne par décision d'un robot d'entraînement, pendant 325 combats (fichier produit par `donnees_apprentissage.py`) ;
- vos combats joués au clavier (bouton « Télécharger mes combats » de l'arène), mêmes colonnes ; la colonne `joueur` vaut 1.

Ce que voit le combattant tient en dix-neuf nombres (`apprentissage.py`) ; la colonne `action` dit ce qu'il a fait.

In [ ]:
import pandas as pd

decisions = pd.read_csv("decisions.csv", sep=";")
# vos combats au clavier, si vous les avez téléchargés dans ce dossier :
# joues = pd.read_csv("combats_joues_alice.csv", sep=";")
# decisions = pd.concat([decisions, joues])
decisions.head()

## 1. Une décision tenue ne compte qu'une fois

Un robot qui tient sa garde pendant 5 secondes décide « garde » 50 fois de suite : 50 lignes identiques. Un arbre qui imite la majorité apprendrait surtout... à rester en garde. Essayez plus tard de sauter cette étape : votre robot 2 ne gagnera plus un seul combat.

On ne garde que les instants où le robot **change** d'action : on compare chaque ligne à la précédente du même robot, dans le même combat (`groupby` puis `shift`).

In [ ]:
print(len(decisions), "lignes avant")
precedente = decisions.groupby(["combat_id", "robot"])["action"].shift()
decisions = decisions[decisions["action"] != precedente]
print(len(decisions), "lignes après")
decisions["action"].value_counts()

## 2. Ne garder que les gagnants

On veut imiter ce qui gagne : les décisions des combattants dont l'issue est `gagne`, et seulement les actions valides (une action refusée n'a rien fait).

In [ ]:
gagnants = decisions[(decisions["issue"] == "gagne") & (decisions["valide"])]
print(len(decisions), "décisions,", len(gagnants), "décisions de gagnants")
gagnants["action"].value_counts()

## 3. Ce que l'arbre voit (`X`) et ce qu'il doit trouver (`y`)

`X` : un tableau, les dix-neuf colonnes que voit le combattant. `y` : une colonne, l'action jouée. On garde un quart des lignes de côté pour tester l'arbre sur des situations qu'il n'a pas vues.

In [ ]:
from sklearn.model_selection import train_test_split
from apprentissage import CARACTERISTIQUES

X = gagnants[CARACTERISTIQUES]
y = gagnants["action"]
X_ent, X_test, y_ent, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
X.shape

## 4. Entraîner l'arbre

On prédit une **catégorie** (une action parmi onze) : c'est une classification. La mesure est le taux de bonnes réponses sur les données de test (`score`). `max_depth=4` : l'arbre pose au plus quatre questions avant de décider.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

arbre = DecisionTreeClassifier(max_depth=4, random_state=0)
arbre.fit(X_ent, y_ent)
print("Bonnes réponses sur le test :", round(arbre.score(X_test, y_test) * 100), "%")

## 5. Lire les règles apprises

`export_text` écrit l'arbre en texte : chaque ligne `|---` est une question, chaque `class:` une action.

In [ ]:
from sklearn.tree import export_text

print(export_text(arbre, feature_names=CARACTERISTIQUES))

## 6. Les règles deviennent votre robot

1. Copiez `robot2_modele.py` en `robot2.py`.
2. La cellule suivante écrit l'arbre en `if` / `else` Python : `|--- distance <= 172.00` devient `if distance <= 172.00:`, et `class: direct` devient `action = "direct"`. Comparez-la ligne à ligne avec l'arbre ci-dessus, puis collez-la dans `robot2.py`, sous la ligne `# COLLEZ ICI` : curseur tout au début de la ligne vide (colonne 1), sans espace devant, car les lignes ont déjà leur retrait.
3. Dans le terminal : `python combat.py --robot robot2.py` (bilan contre les six robots d'entraînement), puis `python combat.py mur --robot robot2.py` pour revoir un combat en 3D.

Pour le championnat : collez `robot2.py` dans l'onglet Programmer de l'arène (génération 2), puis Envoyer.

In [ ]:
from apprentissage import arbre_en_code

print(arbre_en_code(arbre))

## Pour aller plus loin

- Changez `max_depth` (3, puis 6) : le score de test monte-t-il ? Recopiez le nouvel arbre : le robot gagne-t-il plus ? Bien imiter les gagnants, est-ce gagner ?
- Ajoutez vos combats joués au clavier (cellule 1), ou les décisions de la manche 1 (export « Décisions des robots » de la page Tournoi) : votre robot imite-t-il mieux les meilleurs joueurs de la classe ?